# Optimus Demo - Tree-based Model Development Pipeline

In [ ]:
import pandas as pd
from optimus.trainer import Train
from IPython.display import clear_output

df = pd.read_csv('your_data.csv')
X = df.drop(['target', 'sample_type'], axis=1)
y = df['target']
e = df[['sample_type', 'target']]

In [ ]:
spec = {
    'age': 'bestKS',
    'income': 'chiMerge',
    'account_age': 'auto',
    'device_type': 'woeMerge',
    'merchant_category': 'woeMerge',
}

In [ ]:
trainer = Train(
    model_path='./demo_models',
    spec=spec,
    model_type='LR',
    tune_method='BO',
    calibration_method='platt',
    mapping_base={300: 0.5, 600: 0.1, 900: 0.01},
    score_floor=300,
    score_cap=900,
    score_bins=[300, 400, 500, 600, 700, 800, 900],
    iv_threshold=0.02,
    corr_threshold=0.8,
    psi_threshold=0.1
)

trainer.fit(X, y, e)
clear_output()

In [ ]:
performance = trainer.transform(X, y, e)

trainer.write_report(
    performance=performance,
    report_path='./demo_models',
    report_name=f'model_report_{trainer.ts}'
)

performance['scorecard']['test'].head()
performance['feature_importance'].head(10)

In [ ]:
original_ts = trainer.ts

trainer.refit_model(ts=original_ts, trial_index=5)
new_performance = trainer.transform(X, y, e)
new_performance['scorecard']['test'].head()

In [ ]:
ts = '20260201_210105'

predictor = Train(model_path='./demo_models')
predictions = predictor.transform(X, y, e, ts=ts)